In [10]:
!pip install tensorflow sentence-transformers transformers torch scikit-learn joblib


In [11]:
import numpy as np
import joblib
import torch
import tensorflow as tf

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForSequenceClassification


In [12]:
# Load ANN
ann_model = tf.keras.models.load_model(
    "/content/drive/MyDrive/DL/CP/Model/ann_semantic_grader.h5",
    compile=False
)


# Load scaler
scaler = joblib.load("/content/drive/MyDrive/DL/CP/Model/feature_scaler.pkl")

# Load feature order
FEATURES = joblib.load("/content/drive/MyDrive/DL/CP/Model/model_features.pkl")

# Load SBERT
sbert = SentenceTransformer("all-MiniLM-L6-v2")

print("Models Loaded Successfully")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models Loaded Successfully


In [13]:
nli_tokenizer = AutoTokenizer.from_pretrained("roberta-large-mnli")
nli_model = AutoModelForSequenceClassification.from_pretrained("roberta-large-mnli")

nli_model.eval()


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.43G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

In [14]:
def extract_features(student, reference, marks):

    emb_student = sbert.encode(student)
    emb_ref = sbert.encode(reference)

    similarity = cosine_similarity(
        emb_student.reshape(1,-1),
        emb_ref.reshape(1,-1)
    )[0][0]

    distance = np.linalg.norm(emb_student - emb_ref)

    len_student = len(student.split())
    len_ref = max(len(reference.split()),1)

    length_ratio = len_student / len_ref

    coverage = similarity * min(length_ratio,1.0)

    features = np.array([
        similarity,
        distance,
        length_ratio,
        coverage,
        marks
    ]).reshape(1,-1)

    return features, similarity


In [15]:
def nli_inference(student, reference):

    inputs = nli_tokenizer(
        reference,
        student,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        logits = nli_model(**inputs).logits

    probs = torch.softmax(logits, dim=1)[0].numpy()

    labels = ["CONTRADICTION","NEUTRAL","ENTAILMENT"]

    result = labels[np.argmax(probs)]

    return result, probs


In [16]:
def evaluate_answer(question, reference, student, marks):

    print("\n📘 Question:", question)
    print("🧾 Student:", student)

    # -------- SBERT + ANN --------
    features, similarity = extract_features(student, reference, marks)

    # semantic gate
    if similarity < 0.30:
        print("❌ Low semantic relevance → 0 marks")
        return 0.0

    features_scaled = scaler.transform(features)

    ann_score = ann_model.predict(features_scaled)[0][0]

    # -------- NLI --------
    nli_label, probs = nli_inference(student, reference)

    print("🧠 NLI Result:", nli_label)

    # -------- Fusion Logic --------
    if nli_label == "CONTRADICTION":
        ann_score *= 0.3

    elif nli_label == "NEUTRAL":
        ann_score *= 0.7

    elif nli_label == "ENTAILMENT":
        ann_score *= 1.0

    final_score = max(0, min(ann_score, marks))

    print("📊 Final Score:", round(final_score,2), "/", marks)

    return round(final_score,2)


In [17]:
question = "What is Feature Engineering?"

reference = """
Feature engineering is the process of selecting and creating
relevant features from raw data to improve machine learning models.
"""

students = [
    "Feature engineering improves ML models by transforming data.",
    "It is a database indexing method.",
    "It involves selecting and creating features using domain knowledge."
]

for s in students:
    evaluate_answer(question, reference, s, 5)



📘 Question: What is Feature Engineering?
🧾 Student: Feature engineering improves ML models by transforming data.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 374ms/step
🧠 NLI Result: ENTAILMENT
📊 Final Score: 1.82 / 5

📘 Question: What is Feature Engineering?
🧾 Student: It is a database indexing method.
❌ Low semantic relevance → 0 marks

📘 Question: What is Feature Engineering?
🧾 Student: It involves selecting and creating features using domain knowledge.
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step
🧠 NLI Result: NEUTRAL
📊 Final Score: 1.8 / 5


In [18]:
question = "What is overfitting and underfitting?"

reference = """
Overfitting occurs when high variance and low bias.
Underfitting occurs when high bias and low variance.
"""

student = """
Underfitting occurs when high variance and low bias.
Overfitting occurs when high bias and low variance.
"""

evaluate_answer(question, reference, student, 2)



📘 Question: What is overfitting and underfitting?
🧾 Student: 
Underfitting occurs when high variance and low bias.
Overfitting occurs when high bias and low variance.

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
🧠 NLI Result: CONTRADICTION
📊 Final Score: 0.45 / 2


np.float32(0.45)